# 06. 스태킹: 01 예측 + 사전 특징을 입력으로 받는 태거
04 처럼 점수를 고정 비율로 더하면 실패했고, 05 에서 사전을 **입력**으로 넣자 모델이 언제 따를지를 배워 순이익이 났다.
같은 방식으로 **01 의 예측 글자를 입력**으로 넣어, 태거가 01 의 답을 출발점으로 고칠 곳만 배우게 한다.

| 비교 기준 | 검증 F1 |
|---|---|
| 01 노이즈 채널 | 0.9834 |
| 05 v2 태거 (처음부터) | 0.9758 |
| 행별로 01·v2 중 나은 쪽 (상한 참고) | 0.9870 |

- 학습 문장의 01 예측은 **5분할 교차 예측**: 01 을 나머지 4/5 로 만들어 1/5 을 예측 (정답 누출 방지)
- prep 출력의 '학습 문장 교차 예측 F1' 과 'val 01 F1' 이 비슷해야 학습·검증 조건이 맞는 것
- 05 의 대책(새 노이즈 샘플, 특징 끄기) 그대로 + 01 특징도 20% 확률로 끄기

**GPU 런타임, 첫 셀부터.** prep 은 CPU 작업이라 CPU 코어가 많은 런타임(L4·A100 = 12코어)이 훨씬 빠르다 (T4 는 2코어).

## 0) 준비: 드라이브 → 로컬 디스크 복사

In [ ]:
import os
os.chdir('/content')
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DRIVE = '/content/drive/MyDrive/데이콘/난독화'
!mkdir -p /content/work && rsync -a --exclude '.playwright-mcp' --exclude '__pycache__' --exclude 'open.zip' "$DRIVE/" /content/work/
%cd /content/work
!ls outputs | grep -E 'pkl|best_params|tagger_v2_best' ; nproc; nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
def save_back():
    if not os.path.exists(DRIVE):
        drive.mount('/content/drive', force_remount=True)
    # v2 사전 캐시(수백 MB)는 제외, 01 교차 예측 캐시(v3_nc_*.pkl)는 다시 만들기 오래 걸려서 드라이브에 백업
    !rsync -a --exclude 'v2_cache_*.pkl' /content/work/outputs/ "$DRIVE/outputs/"
    print('saved outputs -> drive')

## 1) prep: 사전 특징 + 01 교차 예측
- 사전 특징 캐시(`v2_cache_split.pkl`)가 없으면 먼저 만든다 (T4·2코어 약 10분)
- 01 교차 예측: 문장 약 1만 x 2벌 디코딩 (12코어 약 5분, 2코어 약 30분 예상)
- 01 교차 예측 캐시는 드라이브에 백업되므로 런타임이 바뀌어도 다시 안 만들어도 됨 (사전 캐시는 다시 필요)

In [ ]:
!python -u stacker.py --mode prep
save_back()

## 2) 학습 (처음부터, 30 epoch)
05 에서 이어 학습보다 처음부터가 새 특징을 더 잘 활용했으므로 처음부터를 기본으로.

In [ ]:
!python -u stacker.py --mode train --epochs 30
save_back()

### (비교용) v2 태거에서 이어 학습
`--run_name warm` 으로 따로 저장되어 위 결과를 덮어쓰지 않음. 주석을 지우고 실행.

In [ ]:
!python -u stacker.py --mode train --init_ckpt outputs/tagger_v2_best_scratch.pt --epochs 15 --lr 2e-4 --run_name warm
save_back()

## 3) 비교

In [ ]:
import pandas as pd
from collections import Counter
res = {}
for name, f in [('01 noisy-channel', 'outputs/val_pred.csv'),
                ('05 v2 scratch', 'outputs/val_pred_tagger_v2_scratch.csv'),
                ('06 stack', 'outputs/val_pred_stack.csv'),
                ('06 stack warm', 'outputs/val_pred_stack_warm.csv')]:
    try:
        d = pd.read_csv(f); d['pred'] = d['pred'].fillna('')
        res[name] = d.set_index('ID'); print(f'{name:18s} F1 {d.f1.mean():.4f}')
    except FileNotFoundError:
        print(f'{name:18s} 파일 없음')
if '06 stack' in res:
    a, b = res['01 noisy-channel'], res['06 stack']
    d = b.f1 - a.f1
    print(f'06 vs 01: 06 이 나은 행 {(d>0).sum()} / 01 이 나은 행 {(d<0).sum()} / 같음 {(d==0).sum()}')
    err = Counter((x, y) for g, p in zip(b.output, b.pred) for x, y in zip(g, p) if x != y)
    print('06 오류 top:', err.most_common(20))
    for i in d.sort_values().index[:3].tolist() + d.sort_values().index[-3:].tolist():
        print(f'\n[{d[i]:+.3f}] IN :', b.loc[i, 'input'][:100]); print('        GT :', b.loc[i, 'output'][:100])
        print('        01 :', a.loc[i, 'pred'][:100]); print('        06 :', b.loc[i, 'pred'][:100])

In [ ]:
r = pd.read_csv('outputs/stack_runs.csv')
r[r.run_id == r.run_id.iloc[-1]][['epoch', 'loss', 'val_f1', 'sec', 'run_name', 'nc_drop', 'feat_drop', 'init_ckpt']]

## 4) test 예측 → `outputs/submission_stack.csv`
test 사전 특징 + 01 디코딩(전체 train 으로 만든 01) 후 예측. 출력의 '01 예측과 다른 문장' 수도 확인.

In [ ]:
!python -u stacker.py --mode predict
save_back()
pd.read_csv('outputs/submission_stack.csv').head()

### (선택) 다 끝나면 런타임 해제 (컴퓨팅 단위 절약)

In [ ]:
#from google.colab import runtime; runtime.unassign()